# Описательные статистики корпусов

In [34]:
import re
import numpy as np
import pandas as pd

# Содержательный токен (словоформа) - непрерывная последовательность кириллических символов
WORD = re.compile(r"[А-Яа-яЁё]+")

def n_words(text):
    return len(WORD.findall(str(text)))


def stats(df, name):
    """Описательные статистики по длине"""
    n_w = df.text.map(n_words)
    q = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    print(f"{name}: {len(df)} документов")
    print(f"медиана {int(n_w.median())}, среднее {n_w.mean():.1f}, минимум {n_w.min()}, максимум {n_w.max()}")
    print("процентили:", ", ".join(f"{p}%={int(v)}" for p, v in zip(q, np.percentile(n_w, q))))
    print("останется в диапазоне:")
    for lo, hi in [(3, 20), (3, 25), (3, 30), (3, 35), (3, 40)]:
        m = n_w.between(lo, hi)
        print(f"    {lo}-{hi}: {m.sum()} ({m.mean():.0%})")
    return n_w

## Комментарии YouTube

In [35]:
yt = pd.read_parquet("./data/raw/yt-comments.parquet")
nw_yt = stats(yt, "YouTube")

YouTube: 53969 документов
медиана 13, среднее 28.0, минимум 0, максимум 839
процентили: 1%=0, 5%=1, 10%=3, 25%=6, 50%=13, 75%=29, 90%=64, 95%=103, 99%=243
останется в диапазоне:
    3-20: 30560 (57%)
    3-25: 33941 (63%)
    3-30: 36354 (67%)
    3-35: 38201 (71%)
    3-40: 39739 (74%)


## Telegram

In [36]:
tg = pd.read_parquet("./data/raw/tg-groups.parquet")
nw_tg = stats(tg, "Telegram")

Telegram: 359525 документов
медиана 7, среднее 10.1, минимум 0, максимум 584
процентили: 1%=0, 5%=1, 10%=2, 25%=4, 50%=7, 75%=12, 90%=20, 95%=28, 99%=59
останется в диапазоне:
    3-20: 277756 (77%)
    3-25: 289962 (81%)
    3-30: 296712 (83%)
    3-35: 300817 (84%)
    3-40: 303432 (84%)


## Отзывы

In [37]:
rr = pd.read_parquet("./data/raw/ru-reviews.parquet")
nw_rr = stats(rr, "RuReviews")

RuReviews: 90000 документов
медиана 13, среднее 19.4, минимум 0, максимум 172
процентили: 1%=0, 5%=2, 10%=4, 25%=7, 50%=13, 75%=25, 90%=42, 95%=57, 99%=100
останется в диапазоне:
    3-20: 56478 (63%)
    3-25: 63785 (71%)
    3-30: 68987 (77%)
    3-35: 72730 (81%)
    3-40: 75622 (84%)


## Оценка объема для референсного корпуса

Сраниваются:

- когерентность случайно составленных тем
- когерентность тем, обученных на рабочей выборке

In [38]:
import pymorphy3
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("russian"))
morph = pymorphy3.MorphAnalyzer()


def to_lemmas(text):
    result = []
    for word in WORD.findall(str(text).lower().replace("ё", "е")):
        if word in STOPWORDS:
            continue
        lemma = morph.parse(word)[0].normal_form.replace("ё", "е")
        if lemma not in STOPWORDS and len(lemma) >= 3:
            result.append(lemma)
    return result

In [39]:
N_TOPICS = 20
BAND = (3, 35)

# самый небольшой корпус
corpus = yt[yt.text.map(n_words).between(*BAND)].reset_index(drop=True)
print(f"в диапазоне {BAND[0]}-{BAND[1]}: {len(corpus)} документов")

doc_lemmas = [to_lemmas(text) for text in corpus.text]

rng = np.random.default_rng(42)
shuffled = rng.permutation(len(corpus))
train_index = shuffled[:10_000]      # для обучения
reference_index = shuffled[10_000:]  # подсчет когерентности

train_texts = [" ".join(doc_lemmas[i]) or "пусто" for i in train_index]
reference_docs = [doc_lemmas[i] for i in reference_index if len(doc_lemmas[i]) >= 2]
print(f"обучение {len(train_texts)}, референс {len(reference_docs)}")

в диапазоне 3-35: 38201 документов
обучение 10000, референс 27553


In [40]:
vectorizer = TfidfVectorizer(min_df=5, max_df=0.4)
matrix = vectorizer.fit_transform(train_texts)
vocabulary = np.array(vectorizer.get_feature_names_out())

model = NMF(N_TOPICS, init="nndsvd", max_iter=400, random_state=42).fit(matrix)
trained_topics = [list(vocabulary[row.argsort()[::-1][:10]]) for row in model.components_]

In [41]:
def coherence(topics, reference, dictionary):
    known_words = set(dictionary.token2id)
    filtered = [[w for w in topic if w in known_words] for topic in topics]
    filtered = [topic for topic in filtered if len(topic) >= 2]
    per_topic = CoherenceModel(topics=filtered, texts=reference, dictionary=dictionary, coherence="c_npmi").get_coherence_per_topic()
    return float(np.mean([v for v in per_topic if np.isfinite(v)]))


def random_topics(dictionary, count=50):
    rng = np.random.default_rng(42)
    word_ids = np.array(list(dictionary.cfs.keys()))
    freq = np.array([dictionary.cfs[i] for i in word_ids], float)
    freq = freq / freq.sum()
    return [[dictionary[i] for i in rng.choice(word_ids, 10, replace=False, p=freq)] for _ in range(count)]

In [42]:
print("референс/случайные/обученные/разница")
for size in [5_000, 10_000, 15_000, 20_000, 25_000]:
    if size > len(reference_docs):
        continue
    reference = reference_docs[:size]
    dictionary = Dictionary(reference)
    random_score = coherence(random_topics(dictionary), reference, dictionary)
    trained_score = coherence(trained_topics, reference, dictionary)
    print(f"{size}/{random_score}/{trained_score}/{trained_score - random_score}")

референс/случайные/обученные/разница
5000/-0.4386701592203112/-0.17690578794766362/0.2617643712726476
10000/-0.38898095172737435/-0.09140333932656704/0.2975776124008073
15000/-0.36487296770788824/-0.05512507361153172/0.3097478940963565
20000/-0.35769477315437576/-0.029603649168471042/0.32809112398590473
25000/-0.342781811790441/-0.018106549456433956/0.324675262334007


In [43]:
def train_topics(doc_index, seed=42):
    texts = [" ".join(doc_lemmas[i]) or "пусто" for i in doc_index]
    vectorizer = TfidfVectorizer(min_df=5, max_df=0.4)
    matrix = vectorizer.fit_transform(texts)
    vocabulary = np.array(vectorizer.get_feature_names_out())
    model = NMF(N_TOPICS, init="nndsvd", max_iter=400, random_state=seed).fit(matrix)
    return [set(vocabulary[row.argsort()[::-1][:10]]) for row in model.components_]


# две разные выборки одного размера - совпадение тем

print("размер выборки/совпадение тем")
for size in [1_000, 2_000, 5_000, 10_000, 15_000]:
    if 2 * size > len(corpus):
        continue
    first = train_topics(shuffled[:size])
    second = train_topics(shuffled[size:2 * size])
    overlap = np.mean([max(len(a & b) for b in second) / 10 for a in first])
    print(f"{size}/{overlap}")

размер выборки/совпадение тем
1000/0.24
2000/0.26999999999999996
5000/0.35000000000000003
10000/0.44499999999999995
15000/0.4800000000000001
